# Diet Problem

Recall the Diet Problem, where the goal is to select a set of foods that will satisfy a set of
daily nutritional requirements at minimum cost. A data instance for this problem is provided in q3.xlsx
on Moodle. Model and solve this problem using the Gurobi solver. Display the objective function value,
all the decision variables’ names and values.

- $n$ is the number of food items, $i \in N=\{1,2,...,n\}$
  
- $m$ is the number of nutrients to be considered, $j \in M=\{1,2,...,m\}$

- $x_i$ is the amount of $j_{th}$ food , $\in \mathbb{Z}^+$

- $n_{ij}$ is the amount of $j_{th}$ nutrient in $i_{th}$ food,

- $N_j^L$ is the lower limit of $j_{th}$ nutrient

- $N_j^U$ is the upper limit of $j_{th}$ nutrient

Then, the formulation is the following:
$$\begin{align}
\min \quad & \sum_{i\in N} c_{i} x_{i} \\
\text{s.t.} \quad & N_j^L \leq \sum_{i\in N} n_{ij} x_{i} \leq N_j^U  && \forall j \in M \\
\end{align}$$

In [2]:
import numpy as np 
import pandas as pd 

In [3]:
with pd.ExcelFile("2diet.xlsx") as xls:
    df1 = pd.read_excel(xls, 'NContent')
    df2 = pd.read_excel(xls, 'NReq')
df1

,Name,Calories,Cholesterol,Total_Fat,Sodium,Carbohydrates,Dietary_Fiber,Protein,Vit_A,Vit_C,Calcium,Iron,Cost (Rs./serving)
0,Carrots,23.7,0.0,0.1,19.2,5.6,1.6,0.6,15471.0,5.1,14.9,0.3,7
1,FCorn,72.2,0.0,0.6,2.5,17.1,2.0,2.5,106.6,5.2,3.3,0.3,18
2,Lettuce,2.6,0.0,0.0,1.8,0.4,0.3,0.2,66.0,0.8,3.8,0.1,2
3,Peppers,20.0,0.0,0.1,1.5,4.8,1.3,0.7,467.7,66.1,6.7,0.3,53
4,Potatoes,171.5,0.0,0.2,15.2,39.9,3.2,3.7,0.0,15.6,22.7,4.3,6
5,Tofu,88.2,0.0,5.5,8.1,2.2,1.4,9.4,98.6,0.1,121.8,6.2,31
6,Spaghetti,358.2,0.0,12.3,1237.1,58.3,11.6,8.2,3055.2,27.9,80.2,2.3,78
7,Tomato,25.8,0.0,0.4,11.1,5.7,1.4,1.0,766.3,23.5,6.2,0.6,27
8,Apple,81.4,0.0,0.5,0.0,21.0,3.7,0.3,73.1,7.9,9.7,0.2,24
9,Banana,104.9,0.0,0.5,1.1,26.7,2.7,1.2,92.3,10.4,6.8,0.4,15


In [4]:
df2

,Name,Unit,Min,Max
0,Calories,cal,2000,2250
1,Cholesterol,mg,0,300
2,Total_Fat,g,0,65
3,Sodium,mg,0,2400
4,Carbohydrates,g,0,300
5,Dietary_Fiber,g,25,100
6,Protein,g,50,100
7,Vit_A,IU,5000,50000
8,Vit_C,IU,50,20000
9,Calcium,mg,800,1600


In [5]:
import gurobipy as gp 
from gurobipy import GRB

# Make a model

In [6]:

m = gp.Model('DietProblem')

Set parameter Username
Set parameter LicenseID to value 2750346
Academic license - for non-commercial use only - expires 2026-12-05


# Add a vector of decision variables $x_i$

In [7]:
x = m.addMVar(lb=0, shape= 30, vtype=GRB.INTEGER, name="x")

# Make an array of $n_{ij}$

In [8]:
n_matrix = df1.iloc[:, 1:-1].to_numpy().T

# Make an array of $c_i$

In [9]:
cost = df1.iloc[:,12].to_numpy()

# Make arrays of $N_j^U$ and $N_j^L$

In [10]:
N_U = df2.iloc[:, 3].to_numpy()

N_L = df2.iloc[:, 2].to_numpy()

# Add constraint: N_L <= n_matrix @ x <= N_U

In [11]:

c0 = m.addConstr(N_L <= n_matrix @ x, "Lower_bound_on_nutrients")
c1 = m.addConstr(n_matrix @ x <= N_U, "Upper_bound_on_nutrients")

# Set the Objective function: min cost @ x

In [12]:
m.setObjective(cost @ x, GRB.MINIMIZE)

# Solve

In [13]:
m.optimize()


Gurobi Optimizer version 13.0.0 build v13.0.0rc1 (win64 - Windows 11+.0 (26200.2))

CPU model: Intel(R) Core(TM) 5 120U, instruction set [SSE2|AVX|AVX2]
Thread count: 10 physical cores, 12 logical processors, using up to 12 threads

Optimize a model with 22 rows, 30 columns and 562 nonzeros (Min)
Model fingerprint: 0xdcd1214c
Model has 30 linear objective coefficients
Variable types: 0 continuous, 30 integer (0 binary)
Coefficient statistics:


  Matrix range     [1e-01, 2e+04]
  Objective range  [2e+00, 8e+01]
  Bounds range     [0e+00, 0e+00]
  RHS range        [1e+01, 5e+04]
Presolve removed 5 rows and 0 columns
Presolve time: 0.00s
Presolved: 17 rows, 30 columns, 450 nonzeros
Variable types: 0 continuous, 30 integer (3 binary)

Root relaxation: objective 1.022902e+02, 10 iterations, 0.00 seconds (0.00 work units)

    Nodes    |    Current Node    |     Objective Bounds      |     Work
 Expl Unexpl |  Obj  Depth IntInf | Incumbent    BestBd   Gap | It/Node Time

     0     0  102.29020    0    7          -  102.29020      -     -    0s
H    0     0                     538.0000000  102.29020  81.0%     -    0s
H    0     0                     467.0000000  102.29020  78.1%     -    0s
H    0     0                     414.0000000  102.29020  75.3%     -    0s
H    0     0                     237.0000000  102.29020  56.8%     -    0s
H    0     0                     210.0000000  102.29020  51.3%     -    0s
H    0     0      

# Print the results

In [16]:
if m.status == GRB.OPTIMAL:
    print(f"Optimal objective function value: {m.ObjVal:g}")
    print(f"Decision variables x: {x.X}")
else:
    print("No Optimal Solution found.")

Optimal objective function value: 123
Decision variables x: [ 1. -0. -0. -0.  3. -0. -0. -0. -0. -0. -0. -0. -0.  6.  1. -0. -0.  5.
 -0. -0.  2. -0. -0. -0. -0.  1.  1. -0. -0. -0.]


# The final table with Optimal food quantity to buy

In [ ]:
results_df = pd.DataFrame(df1['Name'].values, columns=['Name'])

results_df['Optimal Quantity'] = x.X

results_df

,Name,Optimal Quantity
0,Carrots,1.0
1,FCorn,-0.0
2,Lettuce,-0.0
3,Peppers,-0.0
4,Potatoes,3.0
5,Tofu,-0.0
6,Spaghetti,-0.0
7,Tomato,-0.0
8,Apple,-0.0
9,Banana,-0.0


In [ ]:
purchased_foods = results_df[results_df['Optimal Quantity'] > 0]
purchased_foods

,Name,Optimal Quantity
0,Carrots,1.0
4,Potatoes,3.0
13,Wheat_Bread,6.0
14,White_Bread,1.0
17,Chocolate_Chip_Cookies,5.0
20,Whole_Milk,2.0
25,White_Rice,1.0
26,Peanut_Butter,1.0
